In [2]:
import numpy as np
import faiss
import time
import os

# Import your custom library
from orange_jumpsuit import IndexPQ as JumpsuitPQ

def read_fvecs(filename):
    """
    Reads a .fvecs file (binary format: [dimension(int32)][vector_elements(float32)]...).
    """
    # Read the whole file as int32
    a = np.fromfile(filename, dtype=np.int32)
    if len(a) == 0:
        return np.empty((0, 0), dtype=np.float32)
    
    # The first element is the dimension
    d = a[0]
    
    # Reshape the array: each row is (1 int for dimension) + (d elements)
    # Then we slice off the first column and view the rest as float32
    return a.reshape(-1, d + 1)[:, 1:].copy().view(np.float32)

def read_ivecs(filename):
    """
    Reads an .ivecs file (binary format: [dimension(int32)][vector_elements(int32)]...).
    Used for the ground truth.
    """
    a = np.fromfile(filename, dtype=np.int32)
    if len(a) == 0:
        return np.empty((0, 0), dtype=np.int32)
    d = a[0]
    return a.reshape(-1, d + 1)[:, 1:].copy()

def recall_at_k(retrieved_indices, groundtruth_indices, k=10):
    """
    Calculates the Recall@k metric.
    Specifically: "For what percentage of queries is the absolute true nearest neighbor 
    (groundtruth_indices[:, 0]) found within the top k retrieved results?"
    """
    top1_hits = 0
    num_queries = len(retrieved_indices)
    
    for i in range(num_queries):
        true_nn = groundtruth_indices[i, 0]
        if true_nn in retrieved_indices[i, :k]:
            top1_hits += 1
            
    return top1_hits / num_queries

In [3]:
# Adjust the path based on your exact folder structure
data_dir = "./data/siftsmall/"
data_name = "siftsmall"

print("Loading data...")
learn_vectors = read_fvecs(os.path.join(data_dir, data_name + "_learn.fvecs"))
base_vectors = read_fvecs(os.path.join(data_dir, data_name + "_base.fvecs"))
query_vectors = read_fvecs(os.path.join(data_dir, data_name + "_query.fvecs"))
groundtruth = read_ivecs(os.path.join(data_dir, data_name + "_groundtruth.ivecs"))

d = base_vectors.shape[1] # Dimension (128 for SIFT)
n_base = base_vectors.shape[0] # Number of database vectors (10,000 for siftsmall)

print(f"Dimension: {d}")
print(f"Base vectors: {n_base}")
print(f"Training vectors: {learn_vectors.shape[0]}")
print(f"Query vectors: {query_vectors.shape[0]}")

Loading data...
Dimension: 128
Base vectors: 10000
Training vectors: 25000
Query vectors: 100


In [4]:
# PQ Parameters
m = 8          # Number of subvectors
nbits = 8      # Number of bits per subvector (2^8 = 256 centroids)
k = 100        # Number of neighbors to retrieve for our search

print("--- FAISS Benchmark ---")

# 1. Initialize
faiss_index = faiss.IndexPQ(d, m, nbits)

# Get the clustering parameters for the PQ
# We want to set 'enforce_reproducible_iterations' or simply a high threshold
faiss_index.pq.cp.niter = 25  # Ensure it's set to your desired count
faiss_index.pq.cp.max_points_per_centroid = 1000000 # Prevent sub-sampling if desired

# To "disable" early stop, set the convergence threshold to 0 or negative
faiss_index.pq.cp.min_it_convergence = 25

# 2. Train
t0 = time.time()
faiss_index.train(learn_vectors)
faiss_train_time = time.time() - t0
print(f"FAISS Train time: {faiss_train_time:.3f} s")

# 3. Add
t0 = time.time()
faiss_index.add(base_vectors)
faiss_add_time = time.time() - t0
print(f"FAISS Add time: {faiss_add_time:.3f} s")

# 4. Search
t0 = time.time()
faiss_distances, faiss_indices = faiss_index.search(query_vectors, k)
faiss_search_time = time.time() - t0
print(f"FAISS Search time: {faiss_search_time:.4f} s")

# 5. Evaluate
faiss_r1 = recall_at_k(faiss_indices, groundtruth, k=1)
faiss_r10 = recall_at_k(faiss_indices, groundtruth, k=10)
faiss_r100 = recall_at_k(faiss_indices, groundtruth, k=100)

print(f"FAISS Recall@1:   {faiss_r1:.4f}")
print(f"FAISS Recall@10:  {faiss_r10:.4f}")
print(f"FAISS Recall@100: {faiss_r100:.4f}")

--- FAISS Benchmark ---
FAISS Train time: 5.508 s
FAISS Add time: 0.276 s
FAISS Search time: 0.1139 s
FAISS Recall@1:   0.4700
FAISS Recall@10:  0.8700
FAISS Recall@100: 1.0000


In [5]:
print("--- Jumpsuit Benchmark ---")

# 1. Initialize
jumpsuit_index = JumpsuitPQ(dimension=d, n_subvectors=m, n_bits_per_value=nbits)

# 2. Train
t0 = time.time()
jumpsuit_index.train(learn_vectors)
jumpsuit_train_time = time.time() - t0
print(f"Jumpsuit Train time: {jumpsuit_train_time:.3f} s")

# 3. Add
t0 = time.time()
jumpsuit_index.add(base_vectors)
jumpsuit_add_time = time.time() - t0
print(f"Jumpsuit Add time: {jumpsuit_add_time:.3f} s")

# 4. Search
t0 = time.time()
jumpsuit_distances, jumpsuit_indices = jumpsuit_index.search(query_vectors, k)
jumpsuit_search_time = time.time() - t0
print(f"Jumpsuit Search time: {jumpsuit_search_time:.4f} s")

# 5. Evaluate
jump_r1 = recall_at_k(jumpsuit_indices, groundtruth, k=1)
jump_r10 = recall_at_k(jumpsuit_indices, groundtruth, k=10)
jump_r100 = recall_at_k(jumpsuit_indices, groundtruth, k=100)

print(f"Jumpsuit Recall@1:   {jump_r1:.4f}")
print(f"Jumpsuit Recall@10:  {jump_r10:.4f}")
print(f"Jumpsuit Recall@100: {jump_r100:.4f}")

--- Jumpsuit Benchmark ---
Jumpsuit Train time: 7.999 s
Jumpsuit Add time: 0.174 s
Jumpsuit Search time: 0.1674 s
Jumpsuit Recall@1:   0.4600
Jumpsuit Recall@10:  0.8300
Jumpsuit Recall@100: 1.0000


In [6]:
import pandas as pd

results = {
    "Metric": ["Train Time (s)", "Add Time (s)", "Search Time (s)", "Recall@1", "Recall@10", "Recall@100"],
    "FAISS": [faiss_train_time, faiss_add_time, faiss_search_time, faiss_r1, faiss_r10, faiss_r100],
    "Jumpsuit": [jumpsuit_train_time, jumpsuit_add_time, jumpsuit_search_time, jump_r1, jump_r10, jump_r100]
}

df = pd.DataFrame(results)
df.set_index("Metric", inplace=True)

print("=== FINAL COMPARISON ===")
display(df.round(4))

=== FINAL COMPARISON ===


,FAISS,Jumpsuit
Metric,,
Train Time (s),5.5077,7.9994
Add Time (s),0.2758,0.1743
Search Time (s),0.1139,0.1674
Recall@1,0.4700,0.4600
Recall@10,0.8700,0.8300
Recall@100,1.0000,1.0000
